# Tarea 2 — Parte 1: Corpus y extracción (OCR + ASR)
**¿Pueden convertir documentos escaneados y entrevistas en audio en un archivo de texto estructurado y buscable?**

**Fecha de extracción:** se imprime en la celda siguiente (fecha de ejecución).

### Fuentes y licencias
| Corpus | Fuente | Licencia / permiso | Acceso |
|---|---|---|---|
| Documentos impresos (35) | Banco Central de Reserva del Perú — *Reporte de Inflación* (marzo, junio, setiembre, diciembre; 2018‑2026) | Publicación institucional de acceso público; se cita la fuente (BCRP). Las imágenes no se redistribuyen: el dashboard enlaza la URL oficial | `https://www.bcrp.gob.pe/docs/Publicaciones/Reporte-Inflacion/{año}/{mes}/reporte-de-inflacion-{mes}-{año}.pdf` |
| Fotos con sombra/inclinación (8) | Capturas propias con celular de páginas del Reporte de Inflación | Fotos propias, sin datos personales | `datos/raw/documentos/fotos/` |
| Manuscritos (9) | Imágenes de manuscritos descargadas de colecciones públicas; fuente y licencia de cada una en `datos/raw/documentos/manuscritos/fuentes_manuscritos.csv` (copiadas a `datos/fuentes.csv`) | Dominio público / CC según cada archivo | URL indicada por archivo |
| Audio (≥30 min, 24 clips) | OpenSLR 73 — *Crowdsourced high-quality Peruvian Spanish speech data set* (Google, 2019). Clips de ~80 s formados uniendo frases de un mismo hablante; 2 clips alternan dos hablantes (multihablante) | CC BY‑SA 4.0 — Guevara‑Rukoz et al. (2020), LREC | `https://www.openslr.org/resources/73/` |

**Modelos:** PaddleOCR 3.x (`lang=es`, PP‑OCRv5 mobile det) · faster‑whisper `small` (CPU int8, VAD on) · diarización ligera MFCC+F0 con clustering aglomerativo. Versiones exactas en la celda siguiente.

In [1]:
import datetime, platform, importlib.metadata as md
import pandas as pd, numpy as np
from pathlib import Path
import t2lib as T

print("Fecha de extracción:", datetime.date.today().isoformat())
print("Python", platform.python_version())
for p in ["paddleocr", "paddlepaddle", "faster-whisper", "pypdfium2", "librosa", "scikit-learn", "plotly"]:
    try: print(f"{p}: {md.version(p)}")
    except md.PackageNotFoundError: print(f"{p}: no instalado")
pd.set_option("display.max_colwidth", 120)

Fecha de extracción: 2026-10-26
Python 3.12.14
paddleocr: 3.3.0
paddlepaddle: 3.2.0
faster-whisper: 1.2.0
pypdfium2: 4.30.0
librosa: 0.11.0
scikit-learn: 1.9.1
plotly: 6.3.0


In [2]:

import plotly.express as px
import plotly.io as pio

def mostrar(fig):
    """Muestra la figura interactiva y además como PNG (para que se vea en GitHub)."""
    try:
        fig.show(renderer="notebook_connected+png")
    except Exception:
        try:
            import kaleido; kaleido.get_chrome_sync()   # descarga Chrome para kaleido 1.x
            fig.show(renderer="notebook_connected+png")
        except Exception as e:
            print("(PNG no disponible:", e, ")")
            fig.show(renderer="notebook_connected")


## 1. Documentos impresos: 35 Reportes de Inflación del BCRP
Descarga respetuosa (User‑Agent descriptivo y pausa de 1,5 s). De cada reporte se rasteriza a 200 dpi la primera página de prosa (≥1 800 letras) y se guarda la capa de texto del PDF como borrador para la referencia manual.

In [4]:
import pypdfium2 as pdfium

def pdf_valido(p):
    try:
        with open(p, "rb") as f:
            if f.read(5) != b"%PDF-":
                return False
        pdfium.PdfDocument(str(p)); return True
    except Exception:
        return False

# 1) borrar los "PDF" falsos (páginas HTML de reportes no publicados)
malos = [p.name for p in T.RAW_BCRP.glob("*.pdf") if not pdf_valido(p)]
for n in malos: (T.RAW_BCRP / n).unlink()
print("Archivos falsos borrados:", malos)

# 2) descarga que rechaza lo que no sea PDF
_desc_original = T.descargar
def descargar_seguro(url, destino, timeout=120):
    ok = _desc_original(url, destino, timeout)
    if ok and destino.suffix == ".pdf" and not pdf_valido(destino):
        destino.unlink(); return False
    return ok
T.descargar = descargar_seguro

ri = T.descargar_reportes_inflacion(n=35)
print(len(ri), "reportes válidos")

# 3) páginas para OCR
docs = []
(T.REFS / "borradores").mkdir(exist_ok=True)
(T.REFS / "para_fotografiar").mkdir(exist_ok=True)
for _, r in ri.iterrows():
    pdf = T.BASE / r.pdf
    try:
        pag = T.elegir_pagina_texto(pdf)
    except Exception as e:
        print("descartado:", pdf.name, e); continue
    did = f"doc_{len(docs)+1:03d}"
    img = T.RAW_DOCS / f"{did}.png"
    txt = T.renderizar_pagina(pdf, pag, img)
    (T.REFS / "borradores" / f"{did}_pdf.txt").write_text(txt, encoding="utf-8")
    if len(docs) < 8:
        g = T.REFS / "para_fotografiar" / f"foto_{len(docs)+1:02d}.png"
        txt_g = T.renderizar_pagina(pdf, pag + 1, g)
        (T.REFS / "para_fotografiar" / f"foto_{len(docs)+1:02d}_pdf.txt").write_text(txt_g, encoding="utf-8")
    docs.append({"id": did, "archivo": str(img.relative_to(T.BASE)).replace("\\", "/"), "tipo": "impreso",
                 "fuente": f"BCRP — Reporte de Inflación {r.mes} {r.anio} (p. {pag+1}), acceso público con cita",
                 "fuente_url": r.url, "pagina": pag + 1})
print(len(docs), "páginas impresas listas")

Archivos falsos borrados: ['RI_2018_diciembre.pdf', 'RI_2018_junio.pdf', 'RI_2018_setiembre.pdf', 'RI_2019_diciembre.pdf', 'RI_2019_junio.pdf', 'RI_2019_marzo.pdf', 'RI_2019_setiembre.pdf', 'RI_2020_diciembre.pdf', 'RI_2020_junio.pdf', 'RI_2020_marzo.pdf', 'RI_2020_setiembre.pdf', 'RI_2021_diciembre.pdf', 'RI_2021_junio.pdf', 'RI_2021_marzo.pdf', 'RI_2021_setiembre.pdf', 'RI_2022_diciembre.pdf', 'RI_2022_junio.pdf', 'RI_2022_marzo.pdf', 'RI_2022_setiembre.pdf', 'RI_2023_diciembre.pdf', 'RI_2023_junio.pdf', 'RI_2023_marzo.pdf', 'RI_2023_setiembre.pdf', 'RI_2024_diciembre.pdf', 'RI_2024_junio.pdf', 'RI_2024_marzo.pdf', 'RI_2024_setiembre.pdf', 'RI_2025_diciembre.pdf', 'RI_2025_junio.pdf', 'RI_2025_marzo.pdf', 'RI_2025_setiembre.pdf', 'RI_2026_diciembre.pdf', 'RI_2026_junio.pdf', 'RI_2026_marzo.pdf', 'RI_2026_setiembre.pdf']
  - no disponible: RI diciembre 2026
  - no disponible: RI setiembre 2026
  - no disponible: RI junio 2026
  - no disponible: RI marzo 2026
  - no disponible: RI dici

## 2. Fotos (8) y manuscritos (9) — capturas propias
Se leen de `datos/raw/documentos/fotos/` y `datos/raw/documentos/manuscritos/`. Se corrige la orientación EXIF y se limita el lado mayor a 2 000 px.

In [6]:
EXT = {".jpg", ".jpeg", ".png", ".heic", ".webp"}
meta_m = T.RAW_DOCS / "manuscritos" / "fuentes_manuscritos.csv"
meta_m = pd.read_csv(meta_m).fillna("").set_index("archivo") if meta_m.exists() else pd.DataFrame()
for tipo, carpeta, fuente in [
        ("foto", "fotos", "Foto propia con celular (sombra/inclinación) de página del Reporte de Inflación BCRP"),
        ("manuscrito", "manuscritos", "Manuscrito descargado")]:
    archivos = sorted(p for p in (T.RAW_DOCS / carpeta).glob("*") if p.suffix.lower() in EXT)
    print(f"{tipo}: {len(archivos)} archivos encontrados")
    for p in archivos:
        did = f"doc_{len(docs)+1:03d}"
        dst = T.preparar_imagen(p, T.RAW_DOCS / f"{did}.jpg")
        if tipo == "manuscrito":
            m = meta_m.loc[p.name] if p.name in meta_m.index else None
            if m is None: print(f"⚠ {p.name}: falta en fuentes_manuscritos.csv")
            fte = f"{m.fuente} ({m.licencia})" if m is not None else "Manuscrito descargado (fuente por completar)"
            url, lic = (m.url, m.licencia) if m is not None else ("", "por completar")
        else:
            fte, url, lic = fuente + " (CC BY 4.0, autora)", "", "CC BY 4.0 (autora)"
        docs.append({"id": did, "archivo": str(dst.relative_to(T.BASE)).replace("\\", "/"), "tipo": tipo,
                     "fuente": fte, "fuente_url": url, "licencia": lic, "origen": p.name})
df_docs = pd.DataFrame(docs)
print(df_docs.tipo.value_counts())
faltan = {"foto": 8, "manuscrito": 9}
for t, n in faltan.items():
    if (df_docs.tipo == t).sum() < n:
        print(f"⚠ Faltan {t}s: hay {(df_docs.tipo == t).sum()}, mínimo {n}")

foto: 8 archivos encontrados
manuscrito: 9 archivos encontrados
⚠ manuscrito_01.jpg: falta en fuentes_manuscritos.csv
⚠ manuscrito_02.jpg: falta en fuentes_manuscritos.csv
⚠ manuscrito_03.jpg: falta en fuentes_manuscritos.csv
⚠ manuscrito_04.jpg: falta en fuentes_manuscritos.csv
⚠ manuscrito_05.jpg: falta en fuentes_manuscritos.csv
⚠ manuscrito_06.jpg: falta en fuentes_manuscritos.csv
⚠ manuscrito_07.jpg: falta en fuentes_manuscritos.csv
⚠ manuscrito_08.jpg: falta en fuentes_manuscritos.csv
⚠ manuscrito_09.jpg: falta en fuentes_manuscritos.csv
tipo
manuscrito    18
foto          16
Name: count, dtype: int64


## 3. Limpieza: duplicados por hash

In [7]:
df_docs["sha1"] = [T.sha1_archivo(T.BASE / a) for a in df_docs.archivo]
dups = df_docs[df_docs.duplicated("sha1")]
print("Duplicados descartados:", len(dups), list(dups.id))
df_docs = df_docs.drop_duplicates("sha1").reset_index(drop=True)

Duplicados descartados: 17 ['doc_018', 'doc_019', 'doc_020', 'doc_021', 'doc_022', 'doc_023', 'doc_024', 'doc_025', 'doc_026', 'doc_027', 'doc_028', 'doc_029', 'doc_030', 'doc_031', 'doc_032', 'doc_033', 'doc_034']


## 4. OCR con PaddleOCR (Sesión 10)
Parámetros: `lang='es'` (modelo de reconocimiento latino PP‑OCRv5), detector `PP-OCRv5_mobile_det`, sin clasificador de orientación ni *unwarping* (configuración base, sin ajuste). El resultado crudo (texto + confianza por línea) se guarda en `datos/raw/documentos/ocr_raw/`.

In [9]:
import json
crudo_dir = T.RAW_DOCS / "ocr_raw"; crudo_dir.mkdir(exist_ok=True)
textos, tiempos = [], []
for _, d in df_docs.iterrows():
    cache = crudo_dir / f"{d.id}.json"
    if cache.exists():
        c = json.loads(cache.read_text(encoding="utf-8"))
    else:
        texto, seg, lineas = T.ocr_paddle(T.BASE / d.archivo)
        c = {"texto": texto, "seg": seg, "lineas": lineas}
        cache.write_text(json.dumps(c, ensure_ascii=False), encoding="utf-8")
    textos.append(c["texto"]); tiempos.append(c["seg"])
    print(f"{d.id} [{d.tipo}] {len(c['texto'])} caracteres, {c['seg']:.1f} s")
df_docs["texto_ocr"] = textos
df_docs["seg_ocr"] = tiempos
df_docs["modelo_ocr"] = "PaddleOCR " + md.version("paddleocr") + ' (lang=es, PP-OCRv5 mobile det)'

doc_001 [foto] 347 caracteres, 31.6 s
doc_002 [foto] 810 caracteres, 49.9 s
doc_003 [foto] 187 caracteres, 22.2 s
doc_004 [foto] 19 caracteres, 10.3 s
doc_005 [foto] 14 caracteres, 5.8 s
doc_006 [foto] 310 caracteres, 20.4 s
doc_007 [foto] 192 caracteres, 23.5 s
doc_008 [foto] 5185 caracteres, 260.4 s
doc_009 [manuscrito] 105 caracteres, 14.1 s
doc_010 [manuscrito] 153 caracteres, 23.7 s
doc_011 [manuscrito] 282 caracteres, 23.0 s
doc_012 [manuscrito] 248 caracteres, 19.8 s
doc_013 [manuscrito] 363 caracteres, 33.6 s
doc_014 [manuscrito] 188 caracteres, 16.2 s
doc_015 [manuscrito] 1210 caracteres, 93.5 s
doc_016 [manuscrito] 489 caracteres, 36.6 s
doc_017 [manuscrito] 74 caracteres, 16.7 s


In [11]:
# Documentos ilegibles: menos de 20 letras reconocidas
df_docs["n_letras"] = df_docs.texto_ocr.str.count(r"[A-Za-zÁÉÍÓÚáéíóúÑñ]")
ilegibles = df_docs[df_docs.n_letras < 20]
print("Ilegibles descartados:", len(ilegibles), list(ilegibles.id), "— motivo: <20 letras reconocidas")
df_docs = df_docs[df_docs.n_letras >= 20].reset_index(drop=True)
df_docs.tipo.value_counts()

Ilegibles descartados: 0 [] — motivo: <20 letras reconocidas


tipo
manuscrito    9
foto          6
Name: count, dtype: int64

## 5. Audio: OpenSLR 73 → clips de ~80 s
Se descargan los dos ZIP (voces femeninas y masculinas). Se unen frases consecutivas de un mismo hablante (pausa de 0,5 s) hasta ~80 s, y 2 clips alternan una voz femenina y una masculina para evaluar diarización. Se guarda el guion de OpenSLR (borrador de referencia).

In [12]:
meta_path = T.RAW_AUDIO / "clips_meta.csv"
if meta_path.exists():
    clips = pd.read_csv(meta_path)
else:
    carpetas = T.descargar_openslr73()
    utt = T.leer_openslr(carpetas)
    print(len(utt), "frases,", utt.hablante.nunique(), "hablantes")
    clips = T.construir_clips(utt, n_mono=22, n_multi=2)
    clips.to_csv(meta_path, index=False)
clips["sha1"] = [T.sha1_archivo(T.BASE / a) for a in clips.archivo_audio]
n0 = len(clips)
clips = clips.drop_duplicates("sha1")
clips["con_voz"] = [T.tiene_voz(T.BASE / a) for a in clips.archivo_audio]
print("Duplicados descartados:", n0 - len(clips), "| sin voz descartados:", (~clips.con_voz).sum())
clips = clips[clips.con_voz].reset_index(drop=True)
print(f"{len(clips)} clips, {clips.duracion_seg.sum()/60:.1f} minutos")

  descargando es_pe_female.zip desde https://www.openslr.org/resources/73/ ...
  descargando es_pe_male.zip desde https://www.openslr.org/resources/73/ ...


KeyboardInterrupt: 

## 6. ASR con faster‑whisper `small` + VAD + diarización (Sesión 13)
Exporta por clip CSV (segmentos), SRT y TXT en `datos/raw/audio/asr/`. La diarización usa MFCC (media y desviación) + F0 mediana por segmento y clustering aglomerativo; el número de hablantes se elige por coeficiente de silueta (1 si la silueta < 0,30).

In [ ]:
asr_dir = T.RAW_AUDIO / "asr"; asr_dir.mkdir(exist_ok=True)
filas = []
for _, c in clips.iterrows():
    cache = asr_dir / f"{c.id}.json"
    if cache.exists():
        r = json.loads(cache.read_text(encoding="utf-8"))
    else:
        segs, seg = T.asr_whisper(T.BASE / c.archivo_audio, "small")
        segs = T.diarizar(T.BASE / c.archivo_audio, segs)
        T.exportar_asr(c.id, segs, asr_dir)
        r = {"segs": segs, "seg": seg}
        cache.write_text(json.dumps(r, ensure_ascii=False), encoding="utf-8")
    n_h = len({s["hablante"] for s in r["segs"]}) or 1
    filas.append({"id": c.id, "texto_asr": T.texto_con_hablantes(r["segs"]) if r["segs"] else "",
                  "n_hablantes": n_h, "seg_asr": r["seg"]})
    print(f"{c.id}: {c.duracion_seg:.0f} s audio, {r['seg']:.1f} s proceso, {n_h} hablante(s) (fuente: {c.n_hablantes_fuente})")
df_aud = clips.merge(pd.DataFrame(filas), on="id")
df_aud["idioma"] = "es (español peruano, OpenSLR 73)"
df_aud["modelo_asr"] = "faster-whisper small (CPU int8, VAD on, beam 5) + diarización MFCC/F0"
df_aud["rtf_asr"] = df_aud.seg_asr / df_aud.duracion_seg
print("Exactitud de n_hablantes vs. fuente:", (df_aud.n_hablantes == df_aud.n_hablantes_fuente).mean().round(2))

## 7. Seudonimización
Se reemplazan DNI (8 dígitos), teléfonos móviles, correos y nombres precedidos de tratamiento (Sr., Sra., Don…). Los nombres de funcionarios citados en su cargo público en los reportes del BCRP no son datos personales privados. **Después se revisó manualmente** una muestra de textos (ver Parte 2).

In [ ]:
tot = {}
for df, col in [(df_docs, "texto_ocr"), (df_aud, "texto_asr")]:
    res = df[col].apply(T.seudonimizar)
    df[col] = res.str[0]
    df["n_reemplazos"] = res.str[1]
    tot[col] = int(df.n_reemplazos.sum())
print("Entidades reemplazadas:", tot)

## 8. Análisis exploratorio (Plotly)

In [ ]:
conteo = df_docs.tipo.value_counts().rename_axis("tipo").reset_index(name="n")
fig = px.bar(conteo, x="tipo", y="n", text="n", color="tipo",
             title=f"Los impresos dominan el corpus ({conteo.n.max()} de {conteo.n.sum()}); fotos y manuscritos cubren los casos difíciles")
mostrar(fig)

In [ ]:
fig = px.bar(df_aud.sort_values("duracion_seg"), x="id", y="duracion_seg", color="n_hablantes_fuente",
             labels={"duracion_seg": "duración (s)", "n_hablantes_fuente": "hablantes"},
             title=f"Los {len(df_aud)} clips duran entre {df_aud.duracion_seg.min():.0f} y {df_aud.duracion_seg.max():.0f} s y suman {df_aud.duracion_seg.sum()/60:.1f} min")
mostrar(fig)

In [ ]:
largos = pd.concat([
    pd.DataFrame({"longitud": df_docs.texto_ocr.str.len(), "medida": "caracteres OCR (" + df_docs.tipo + ")"}),
    pd.DataFrame({"longitud": df_aud.texto_asr.str.split().str.len(), "medida": "palabras ASR por clip"})])
med = df_docs.groupby("tipo").texto_ocr.apply(lambda s: s.str.len().median())
fig = px.histogram(largos, x="longitud", color="medida", facet_row=largos.medida.str.startswith("palabras"),
                   nbins=30, title=f"Los manuscritos producen textos mucho más cortos (mediana {med.get('manuscrito', 0):.0f} caracteres) que los impresos ({med.get('impreso', 0):.0f})")
fig.update_yaxes(matches=None); fig.for_each_annotation(lambda a: a.update(text=""))
mostrar(fig)

## 9. Validación: extremos de longitud

In [ ]:
df_docs["len_ocr"] = df_docs.texto_ocr.str.len()
print("3 documentos con OCR más largo:"); display(df_docs.nlargest(3, "len_ocr")[["id", "tipo", "len_ocr", "texto_ocr"]])
print("3 documentos con OCR más corto:"); display(df_docs.nsmallest(3, "len_ocr")[["id", "tipo", "len_ocr", "texto_ocr"]])
print("3 clips más largos:"); display(df_aud.nlargest(3, "duracion_seg")[["id", "duracion_seg", "texto_asr"]])
print("3 clips más cortos:"); display(df_aud.nsmallest(3, "duracion_seg")[["id", "duracion_seg", "texto_asr"]])

## 10. Ejemplos publicables y exportación
Se copian a `datos/ejemplos/` 4 audios recortados a 30 s (OGG, CC BY‑SA 4.0 con atribución) y las imágenes de fotos y manuscritos propios (900 px). Las páginas del BCRP no se copian: se enlaza la URL oficial.

In [ ]:
import soundfile as sf
from PIL import Image
for cid in list(df_aud.id[:4]):
    x = T._cargar_16k(T.BASE / df_aud.set_index("id").loc[cid, "archivo_audio"])[:30 * 16000]
    sf.write(T.EJEMPLOS / f"{cid}.ogg", x, 16000, format="OGG", subtype="VORBIS")
libre = df_docs.licencia.fillna("").str.contains(r"CC|dominio|public", case=False)
for _, d in df_docs[(df_docs.tipo != "impreso") & libre].iterrows():
    im = Image.open(T.BASE / d.archivo); im.thumbnail((900, 900)); im.save(T.EJEMPLOS / f"{d.id}.jpg", quality=80)
df_aud["ejemplo_audio"] = [f"datos/ejemplos/{i}.ogg" if (T.EJEMPLOS / f"{i}.ogg").exists() else "" for i in df_aud.id]
df_docs["ejemplo_img"] = [f"datos/ejemplos/{i}.jpg" if (T.EJEMPLOS / f"{i}.jpg").exists() else "" for i in df_docs.id]
print(len(list(T.EJEMPLOS.glob("*"))), "archivos de ejemplo")

In [ ]:
prev = T.DATOS / "documentos.csv"
ref_docs = pd.read_csv(prev).set_index("id").texto_referencia if prev.exists() else pd.Series(dtype=str)
df_docs["texto_referencia"] = df_docs.id.map(ref_docs).fillna("")
cols_d = ["id", "archivo", "tipo", "fuente", "texto_ocr", "modelo_ocr", "texto_referencia",
          "fuente_url", "seg_ocr", "n_reemplazos", "ejemplo_img"]
df_docs[cols_d].to_csv(prev, index=False)

prev = T.DATOS / "transcripciones.csv"
ref_aud = pd.read_csv(prev).set_index("id").texto_referencia if prev.exists() else pd.Series(dtype=str)
df_aud["texto_referencia"] = df_aud.id.map(ref_aud).fillna("")
df_aud["fuente"] = "OpenSLR 73 (CC BY-SA 4.0) hablantes " + df_aud.hablantes_fuente
cols_a = ["id", "archivo_audio", "duracion_seg", "idioma", "texto_asr", "modelo_asr", "n_hablantes",
          "texto_referencia", "fuente", "n_hablantes_fuente", "seg_asr", "rtf_asr", "ejemplo_audio", "guion_openslr"]
df_aud[cols_a].to_csv(prev, index=False)

fuentes = pd.concat([
    pd.DataFrame({"id": df_docs.id, "modalidad": "documento", "fuente_url": df_docs.fuente_url.replace("", "propia"),
                  "licencia": np.where(df_docs.tipo == "impreso", "BCRP, acceso público con cita de fuente", df_docs.licencia.fillna("")),
                  "acceso": np.select([df_docs.tipo == "impreso", df_docs.tipo == "foto"],
                                      ["Descargar el PDF y rasterizar la página indicada en 'fuente' (01_extraccion.ipynb §1)",
                                       "Captura propia; copia ligera en datos/ejemplos/"],
                                      "Descargar desde fuente_url; copia ligera en datos/ejemplos/ si la licencia lo permite")}),
    pd.DataFrame({"id": df_aud.id, "modalidad": "audio", "fuente_url": "https://www.openslr.org/resources/73/",
                  "licencia": "CC BY-SA 4.0 — Guevara-Rukoz et al. (2020)",
                  "acceso": "es_pe_female.zip / es_pe_male.zip; reconstruir con t2lib.construir_clips (semilla 13); hablantes: " + df_aud.hablantes_fuente})])
fuentes.to_csv(T.DATOS / "fuentes.csv", index=False)

for n in ["documentos", "transcripciones", "fuentes"]:
    d = pd.read_csv(T.DATOS / f"{n}.csv"); print(n, d.shape); display(d.head())
print(f"Total: {len(df_docs)} documentos | {df_aud.duracion_seg.sum()/60:.1f} min de audio en {len(df_aud)} clips")